# 04 — Training (MentalBERT Fine-Tuning)

**Project:** MentalBERT-CSSR  
**Goal:** Fine-tune `mental/mental-bert-base-uncased` for 7-class C-SSRS severity classification.

---

## Pipeline

```text
Processed CSV → Stratified Split → MentalBERT Tokenizer (left trunc)
    → DataLoader → MentalBERT Classifier → Train / Validate
    → Metrics → Early Stopping → Checkpoints → Plots / CSV
```

## Design decisions

| Topic | Choice | Why |
|-------|--------|-----|
| Encoder | **MentalBERT only** | Project lock; domain mental-health pretraining |
| Head | `BertForSequenceClassification` on MentalBERT weights | Standard HF classification head; same checkpoint repo |
| Labels | Human `severity` only | LLM columns ignored |
| Split | 70 / 15 / 15 stratified | Preserve rare high-severity classes |
| Truncation | `left`, `max_length` from Notebook 3 | Keep clinical climax / intent |
| Loss | `CrossEntropyLoss` + label smoothing | Soft targets; reduce overconfidence |
| Optimiser | AdamW + weight decay | Standard transformer fine-tuning |
| Schedule | Cosine decay + warmup | Stable early steps, smooth anneal |
| AMP | `torch.cuda.amp` when CUDA | Faster / less memory on GPU |
| Grad clip | `max_grad_norm=1.0` | Stabilise fine-tuning |
| Early stop / best ckpt | Validation **macro-F1** | Minority severity classes are clinically critical |
| Test split | Held out (not used in `fit`) | Reserved for Notebook 5 |

**Stop after this notebook** until Notebook 5 is approved.

## 1. Environment, configuration, reproducibility

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd
import torch

NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = None
for candidate in [NOTEBOOK_DIR, NOTEBOOK_DIR.parent]:
    if (candidate / "configs" / "default.yaml").exists() and (candidate / "utils").exists():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    raise RuntimeError("Could not locate project root.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from utils import ensure_directories, get_logger, load_config, set_seed
from utils.dataset import (
    compute_balanced_class_weights,
    create_dataloader,
    split_label_counts,
    stratified_split,
)
from utils.device import resolve_device
from utils.io import load_processed_dataset, training_columns
from utils.model import load_mentalbert_for_classification
from utils.runtime_settings import resolve_max_length
from utils.tokenization import MENTALBERT_MODEL_NAME, load_mentalbert_tokenizer
from utils.trainer import MentalBERTTrainer, TrainerConfig

pd.set_option("display.max_columns", 50)

cfg = load_config()
set_seed(cfg.SEED)
ensure_directories(cfg)
logger = get_logger("notebook.04_training")

device = resolve_device(cfg.DEVICE)
MAX_LENGTH, max_length_source = resolve_max_length(
    cfg.MAX_LENGTH,
    recommendation_path=cfg.tokenization.recommendation_path,
    project_root=PROJECT_ROOT,
)

assert cfg.MODEL_NAME == MENTALBERT_MODEL_NAME
assert cfg.model.truncation_side == "left"

print(f"Device           : {device}")
print(f"CUDA available   : {torch.cuda.is_available()}")
print(f"Model            : {cfg.MODEL_NAME}")
print(f"max_length       : {MAX_LENGTH} (source={max_length_source})")
print(f"batch_size       : {cfg.BATCH_SIZE} (effective ≈ {cfg.BATCH_SIZE * int(cfg.training.get('gradient_accumulation_steps', 1))})")
print(f"epochs           : {cfg.EPOCHS}")
print(f"lr / wd / warmup : {cfg.LEARNING_RATE} / {cfg.WEIGHT_DECAY} / {cfg.WARMUP_RATIO}")
print(f"head lr mult     : {cfg.training.classifier_lr_mult}")
print(f"loss             : {cfg.training.loss_type} (gamma={cfg.training.focal_gamma})")
print(f"class weights    : {cfg.training.use_class_weights}")
print(f"weighted sampler : {cfg.training.use_weighted_sampler}")
print(f"grad accum       : {cfg.training.gradient_accumulation_steps}")
print(f"freeze encoder   : {cfg.training.freeze_encoder_epochs} epoch(s)")
print(f"label_smoothing  : {cfg.LABEL_SMOOTHING}")
print(f"dropout          : {cfg.DROPOUT}")
print(f"AMP              : {cfg.USE_MIXED_PRECISION}")
print(f"patience         : {cfg.PATIENCE}")
print(f"monitor_metric   : {cfg.training.monitor_metric}")
print(f"seed             : {cfg.SEED}")

## 2. Load processed data (human labels only for training targets)

In [ ]:
df_full = load_processed_dataset(
    cfg.paths.processed_data,
    text_column=cfg.data.text_column,
    label_column=cfg.data.label_column,
)

# Keep a lean supervised matrix; LLM labels are never training targets
df = training_columns(
    df_full,
    text_column=cfg.data.text_column,
    label_column=cfg.data.label_column,
    ignore_columns=list(cfg.data.ignore_columns),
)

print("Shape:", df.shape)
print("Columns:", list(df.columns))
display(df.head(3))
print("Label counts:")
display(df[cfg.data.label_column].value_counts().sort_index())

## 3. Stratified train / validation / test split

Test split is **persisted but not used** during `fit()` — Notebook 5 evaluates the best checkpoint on it.

In [ ]:
train_df, val_df, test_df = stratified_split(
    df,
    text_column=cfg.data.text_column,
    label_column=cfg.data.label_column,
    train_ratio=float(cfg.training.train_ratio),
    val_ratio=float(cfg.training.val_ratio),
    test_ratio=float(cfg.training.test_ratio),
    seed=cfg.SEED,
    stratify=bool(cfg.training.stratify),
)

split_summary = pd.DataFrame([
    {"split": "train", "n": len(train_df), **split_label_counts(train_df, cfg.data.label_column)},
    {"split": "val", "n": len(val_df), **split_label_counts(val_df, cfg.data.label_column)},
    {"split": "test", "n": len(test_df), **split_label_counts(test_df, cfg.data.label_column)},
])
display(split_summary)

if bool(cfg.training.persist_splits):
    splits_dir = PROJECT_ROOT / cfg.training.splits_dir
    splits_dir.mkdir(parents=True, exist_ok=True)
    train_df.to_csv(splits_dir / "train.csv", index=False)
    val_df.to_csv(splits_dir / "val.csv", index=False)
    test_df.to_csv(splits_dir / "test.csv", index=False)
    print(f"Splits saved → {splits_dir}")

## 4. Tokenizer + DataLoaders

MentalBERT WordPiece tokenizer with `truncation_side="left"`.

In [ ]:
tokenizer = load_mentalbert_tokenizer(
    model_name=cfg.MODEL_NAME,
    truncation_side=cfg.model.truncation_side,
)
assert tokenizer.truncation_side == "left"

pin_memory = bool(cfg.runtime.pin_memory) and device.type == "cuda"

class_weights = compute_balanced_class_weights(
    train_df[cfg.data.label_column].astype(int).tolist(),
    num_labels=cfg.NUM_LABELS,
)
display(pd.DataFrame({"severity": list(range(cfg.NUM_LABELS)), "weight": class_weights.numpy()}))

train_loader = create_dataloader(
    train_df,
    tokenizer,
    text_column=cfg.data.text_column,
    label_column=cfg.data.label_column,
    max_length=MAX_LENGTH,
    batch_size=cfg.BATCH_SIZE,
    shuffle=True,
    num_workers=int(cfg.runtime.num_workers),
    pin_memory=pin_memory,
    use_weighted_sampler=bool(cfg.training.use_weighted_sampler),
    num_labels=cfg.NUM_LABELS,
)
val_loader = create_dataloader(
    val_df,
    tokenizer,
    text_column=cfg.data.text_column,
    label_column=cfg.data.label_column,
    max_length=MAX_LENGTH,
    batch_size=cfg.BATCH_SIZE,
    shuffle=False,
    num_workers=int(cfg.runtime.num_workers),
    pin_memory=pin_memory,
)
test_loader = create_dataloader(
    test_df,
    tokenizer,
    text_column=cfg.data.text_column,
    label_column=cfg.data.label_column,
    max_length=MAX_LENGTH,
    batch_size=cfg.BATCH_SIZE,
    shuffle=False,
    num_workers=int(cfg.runtime.num_workers),
    pin_memory=pin_memory,
)

print(f"train batches: {len(train_loader)} | val batches: {len(val_loader)} | test batches: {len(test_loader)}")
sample = next(iter(train_loader))
print({k: tuple(v.shape) for k, v in sample.items()})

## 5. Load MentalBERT classifier

In [ ]:
model = load_mentalbert_for_classification(
    model_name=cfg.MODEL_NAME,
    num_labels=cfg.NUM_LABELS,
    dropout=cfg.DROPOUT,
)

n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parameters: {n_params:,} (trainable {n_trainable:,})")
print(model.config.architectures if hasattr(model.config, "architectures") else model.__class__.__name__)

## 6. Train

Implements CrossEntropy + label smoothing, AdamW, cosine+warmup, AMP, grad clipping, early stopping, checkpointing.

In [ ]:
experiment_metadata = {
    "dataset_name": cfg.experiment.dataset_name,
    "experiment_name": cfg.experiment.name,
    "model_name": cfg.MODEL_NAME,
    "max_length": MAX_LENGTH,
    "max_length_source": max_length_source,
    "truncation_side": cfg.model.truncation_side,
    "num_labels": cfg.NUM_LABELS,
    "epochs": cfg.EPOCHS,
    "learning_rate": cfg.LEARNING_RATE,
    "batch_size": cfg.BATCH_SIZE,
    "dropout": cfg.DROPOUT,
    "weight_decay": cfg.WEIGHT_DECAY,
    "warmup_ratio": cfg.WARMUP_RATIO,
    "label_smoothing": cfg.LABEL_SMOOTHING,
    "loss_type": cfg.training.loss_type,
    "focal_gamma": cfg.training.focal_gamma,
    "use_class_weights": cfg.training.use_class_weights,
    "use_weighted_sampler": cfg.training.use_weighted_sampler,
    "gradient_accumulation_steps": cfg.training.gradient_accumulation_steps,
    "classifier_lr_mult": cfg.training.classifier_lr_mult,
    "freeze_encoder_epochs": cfg.training.freeze_encoder_epochs,
    "patience": cfg.PATIENCE,
    "max_grad_norm": cfg.training.max_grad_norm,
    "use_mixed_precision": cfg.USE_MIXED_PRECISION,
    "optimizer": cfg.training.optimizer,
    "scheduler": cfg.training.scheduler,
    "seed": cfg.SEED,
    "device": str(device),
    "monitor_metric": cfg.training.monitor_metric,
    "train_size": len(train_df),
    "val_size": len(val_df),
    "test_size": len(test_df),
}

trainer_cfg = TrainerConfig(
    learning_rate=cfg.LEARNING_RATE,
    classifier_lr_mult=float(cfg.training.classifier_lr_mult),
    weight_decay=cfg.WEIGHT_DECAY,
    warmup_ratio=cfg.WARMUP_RATIO,
    label_smoothing=cfg.LABEL_SMOOTHING,
    epochs=cfg.EPOCHS,
    patience=cfg.PATIENCE,
    max_grad_norm=float(cfg.training.max_grad_norm),
    use_mixed_precision=bool(cfg.USE_MIXED_PRECISION),
    num_labels=cfg.NUM_LABELS,
    seed=cfg.SEED,
    monitor_metric=str(cfg.training.monitor_metric),
    loss_type=str(cfg.training.loss_type),
    focal_gamma=float(cfg.training.focal_gamma),
    use_class_weights=bool(cfg.training.use_class_weights),
    gradient_accumulation_steps=int(cfg.training.gradient_accumulation_steps),
    freeze_encoder_epochs=int(cfg.training.freeze_encoder_epochs),
    save_dir=Path(cfg.SAVE_PATH),
    results_dir=Path(cfg.RESULTS_PATH),
    plots_dir=Path(cfg.paths.plots_path),
    metrics_dir=Path(cfg.paths.metrics_path),
    model_path=Path(cfg.MODEL_PATH),
    optimizer_path=Path(cfg.SAVE_PATH) / "optimizer.pt",
    figure_dpi=int(cfg.eda.figure_dpi),
)

trainer = MentalBERTTrainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    config=trainer_cfg,
    label_list=list(range(cfg.NUM_LABELS)),
    experiment_metadata=experiment_metadata,
    class_weights=class_weights,
)

history_df = trainer.fit()
display(history_df)

## 7. Training artefacts checklist

In [ ]:
artefacts = [
    Path(cfg.MODEL_PATH),
    Path(cfg.SAVE_PATH) / "optimizer.pt",
    Path(cfg.paths.metrics_path) / "history.csv",
    Path(cfg.paths.metrics_path) / "training_metrics.csv",
    Path(cfg.paths.metrics_path) / "val_classification_report.csv",
    Path(cfg.paths.metrics_path) / "val_best_metrics.json",
    Path(cfg.paths.metrics_path) / "training_experiment.json",
    Path(cfg.paths.plots_path) / "training_loss.png",
    Path(cfg.paths.plots_path) / "validation_loss.png",
    Path(cfg.paths.plots_path) / "training_accuracy.png",
    Path(cfg.paths.plots_path) / "validation_accuracy.png",
    Path(cfg.paths.plots_path) / "macro_f1.png",
    Path(cfg.paths.plots_path) / "micro_f1.png",
    Path(cfg.paths.plots_path) / "weighted_f1.png",
    Path(cfg.paths.plots_path) / "learning_rate.png",
    Path(cfg.paths.plots_path) / "confusion_matrix.png",
]

status = pd.DataFrame([
    {"artefact": str(p), "exists": p.exists(), "size_bytes": p.stat().st_size if p.exists() else 0}
    for p in artefacts
])
display(status)
missing = status.loc[~status["exists"], "artefact"].tolist()
if missing:
    print("WARNING — missing artefacts:")
    for m in missing:
        print(" ", m)
else:
    print("All expected training artefacts present.")

# Sanity: do not silently evaluate test inside training notebook beyond size log
print(f"Held-out test size (for Notebook 5): {len(test_df)}")
print("Test loader ready but NOT evaluated here.")

## 8. Summary → Notebook 5

| Artefact | Path |
|----------|------|
| Best model | `saved_model/best_model.pt` |
| Optimiser | `saved_model/optimizer.pt` |
| History | `RESULTS/metrics/history.csv` |
| Metrics CSV | `RESULTS/metrics/training_metrics.csv` |
| Splits | `DATA/processed/splits/{train,val,test}.csv` |

Notebook 5 must load `best_model.pt`, run the **test** split, and produce error analysis / confidence plots.

---

**Stop here.** Await approval before generating Notebook 5 (Evaluation).